# RoBERTa — External Evaluation

**Environment:** Google Colab | NVIDIA T4 GPU | PyTorch | Hugging Face Transformers

> **Execution note:** This notebook was developed and executed in Google Colab. It evaluates a previously fine-tuned RoBERTa model on a modern dataset containing human-written and AI-generated responses. The model was also fine-tuned in Google Colab.

The notebook requires access to the saved model weights and evaluation dataset. File paths and environment configuration must be adjusted for local execution.

**Model:** `FacebookAI/roberta-base` (fine-tuned on HC3)
**Evaluation dataset:** 600 responses from humans, ChatGPT, Claude, and Gemini
**Related notebook:** `roberta_fine_tuning.ipynb`

## Mount Google Drive

In [1]:

from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/chewy-colab")

print("Project root:", PROJECT_ROOT)
print("Project exists:", PROJECT_ROOT.exists())


Mounted at /content/drive
Project root: /content/drive/MyDrive/chewy-colab
Project exists: True


## Import the libraries

In [2]:

import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("GPU available:", torch.cuda.is_available())


GPU available: True


## Load the modern evaluation dataset

In [3]:

modern_path = PROJECT_ROOT / "processed" / "modern_dataset_drift_eval.csv"

modern = pd.read_csv(modern_path)

print("Dataset shape:", modern.shape)
print("\nColumns:", modern.columns.tolist())

display(modern.head())


Dataset shape: (600, 14)

Columns: ['answer', 'question_id', 'question', 'question_context', 'label', 'model', 'domain', 'source', 'question_date', 'answer_date', 'question_score', 'answer_score', 'is_accepted', 'tags']


,answer,question_id,question,question_context,label,model,domain,source,question_date,answer_date,question_score,answer_score,is_accepted,tags
0,So why is the ocean so clean? You can drink it...,26756,What happens to all the icky stuff?,"All of the most horrible, disgusting liquids I...",human,human,general,earthscience,2025-01-09,2025-01-10,4,3.0,True,"['oceanography', 'groundwater', 'nutrient-cycl..."
1,The ocean is not actually “clean” in the sense...,26756,What happens to all the icky stuff?,"All of the most horrible, disgusting liquids I...",ai,chatgpt,general,earthscience,2025-01-09,2026-10-29,4,NaN,NaN,"['oceanography', 'groundwater', 'nutrient-cycl..."
2,"The ocean isn't as clean as it looks, but you'...",26756,What happens to all the icky stuff?,"All of the most horrible, disgusting liquids I...",ai,claude,general,earthscience,2025-01-09,NaN,4,NaN,NaN,"['oceanography', 'groundwater', 'nutrient-cycl..."
3,Wastewater and organic waste undergo municipal...,26756,What happens to all the icky stuff?,"All of the most horrible, disgusting liquids I...",ai,gemini,general,earthscience,2025-01-09,2025-01-10,4,NaN,NaN,"['oceanography', 'groundwater', 'nutrient-cycl..."
4,"CO2, O3, and CH4, which constitute, in combina...",26764,How do greenhouse gases absorb so much radiati...,I was hoping this forum could give me an intui...,human,human,general,earthscience,2025-01-12,2025-01-13,20,51.0,False,"['greenhouse-gases', 'atmospheric-radiation', ..."


## Prepare the text and labels

In [4]:

modern = modern.rename(columns={
    "answer": "text",
    "question_id": "id",
    "source": "question_source"
})

modern["target"] = modern["label"].map({
    "human": 0,
    "ai": 1
}).astype(int)

print("Total responses:", len(modern))
print("\nResponses by generator:")
print(pd.crosstab(modern["model"], modern["target"]))

display(modern[["id", "model", "label", "target", "text"]].head())


Total responses: 600

Responses by generator:
target     0    1
model            
chatgpt    0  150
claude     0  150
gemini     0  150
human    150    0


,id,model,label,target,text
0,26756,human,human,0,So why is the ocean so clean? You can drink it...
1,26756,chatgpt,ai,1,The ocean is not actually “clean” in the sense...
2,26756,claude,ai,1,"The ocean isn't as clean as it looks, but you'..."
3,26756,gemini,ai,1,Wastewater and organic waste undergo municipal...
4,26764,human,human,0,"CO2, O3, and CH4, which constitute, in combina..."


## Load the fine-tuned RoBERTa model

In [5]:

MODEL_SAVE_DIR = PROJECT_ROOT / "models" / "roberta" / "roberta_384_58k"

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_SAVE_DIR))

model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_SAVE_DIR)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Model loaded from:", MODEL_SAVE_DIR)
print("Running on:", device)
print("Number of labels:", model.config.num_labels)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded from: /content/drive/MyDrive/chewy-colab/models/roberta/roberta_384_58k
Running on: cuda
Number of labels: 2


## Tokenize the modern dataset

In [6]:

from datasets import Dataset

modern_dataset = Dataset.from_pandas(
    modern[["text", "target"]].rename(columns={"target": "labels"}),
    preserve_index=False
)

def tokenize_modern(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=384
    )

tokenized_modern = modern_dataset.map(
    tokenize_modern,
    batched=True
)

print("Tokenized responses:", len(tokenized_modern))
print("First response token count:", len(tokenized_modern[0]["input_ids"]))


Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Tokenized responses: 600
First response token count: 384


## Run RoBERTa on all 600 modern responses

In [8]:

from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenized_modern = tokenized_modern.remove_columns(["text"])

modern_loader = DataLoader(
    tokenized_modern,
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator
)

all_logits = []

with torch.inference_mode():
    for batch in modern_loader:
        batch = {
            k: v.to(device)
            for k, v in batch.items()
            if k in ["input_ids", "attention_mask"]
        }

        outputs = model(**batch)
        all_logits.append(outputs.logits.cpu())

logits = torch.cat(all_logits).numpy()

modern_scores = softmax(logits, axis=1)[:, 1]
modern_predictions = np.argmax(logits, axis=1)

print("Predictions generated:", len(modern_predictions))
print("Predicted human:", (modern_predictions == 0).sum())
print("Predicted AI:", (modern_predictions == 1).sum())


Predictions generated: 600
Predicted human: 554
Predicted AI: 46


## Calculate evaluation metrics

In [9]:

from sklearn.metrics import precision_score, recall_score

y_true = modern["target"].to_numpy()

metrics = {
    "accuracy": accuracy_score(y_true, modern_predictions),
    "macro_precision": precision_score(y_true, modern_predictions, average="macro", zero_division=0),
    "macro_recall": recall_score(y_true, modern_predictions, average="macro", zero_division=0),
    "macro_f1": f1_score(y_true, modern_predictions, average="macro", zero_division=0),
    "roc_auc": roc_auc_score(y_true, modern_scores),
    "human_false_positive_rate": float(modern_predictions[y_true == 0].mean())
}

for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, modern_predictions))


accuracy: 0.3200
macro_precision: 0.6118
macro_recall: 0.5422
macro_f1: 0.2989
roc_auc: 0.7088
human_false_positive_rate: 0.0133

Confusion Matrix:
[[148   2]
 [406  44]]


## See which AI generators RoBERTa struggles with

In [10]:

modern["roberta_score"] = modern_scores
modern["roberta_prediction"] = modern_predictions

generator_results = modern.groupby("model").agg(
    total=("target", "size"),
    actual_ai=("target", "sum"),
    predicted_ai=("roberta_prediction", "sum"),
    avg_ai_score=("roberta_score", "mean")
)

generator_results["ai_prediction_rate"] = (
    generator_results["predicted_ai"] / generator_results["total"]
)

display(generator_results.round(4))


,total,actual_ai,predicted_ai,avg_ai_score,ai_prediction_rate
model,,,,,
chatgpt,150,150,14,0.0941,0.0933
claude,150,150,28,0.1704,0.1867
gemini,150,150,2,0.0133,0.0133
human,150,0,2,0.0140,0.0133


In [ ]:

roberta_generator_export = pd.read_csv(
    PROJECT_ROOT / "data" / "evaluation" /
    "modern_roberta_generator_metrics.csv"
)

roberta_generator_export = roberta_generator_export.rename(
    columns={
        "total": "n",
        "ai_prediction_rate": "rate"
    }
)

roberta_generator_export["model"] = "RoBERTa"

roberta_generator_export["metric"] = np.where(
    roberta_generator_export["generator"].str.lower() == "human",
    "human_false_positive_rate",
    "ai_recall"
)

roberta_generator_export = roberta_generator_export[
    ["model", "generator", "n", "metric", "rate",
     "actual_ai", "predicted_ai", "avg_ai_score"]
]

roberta_generator_export.to_csv(
    PROJECT_ROOT / "data" / "evaluation" /
    "modern_roberta_generator_metrics.csv",
    index=False
)


## Save the modern evaluation results

In [11]:

EVAL_DIR = PROJECT_ROOT / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Save model-level metrics
pd.DataFrame([{
    "model": "RoBERTa",
    **metrics
}]).to_csv(
    EVAL_DIR / "modern_roberta_metrics.csv",
    index=False
)

# Save predictions for all 600 responses
modern[[
    "id", "model", "label", "target",
    "roberta_score", "roberta_prediction"
]].to_csv(
    EVAL_DIR / "modern_roberta_predictions.csv",
    index=False
)

# Save generator-level results
generator_results.to_csv(
    EVAL_DIR / "modern_roberta_generator_metrics.csv"
)

print("Saved evaluation files to:", EVAL_DIR)


Saved evaluation files to: /content/drive/MyDrive/chewy-colab/evaluation
